# Sahte Haber Tespiti

Bu projede haber metninin FAKE mi REAL mi olduğunu tahmin edeceğim.


In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns


### Data


In [ ]:
df=pd.read_csv('data/fake_or_real_news.csv')
df.head()


### EDA


In [ ]:
df.info()
df.isnull().sum()


In [ ]:
df['label'].value_counts()


### Görselleştirme


In [ ]:
df['uzunluk']=df['text'].astype(str).str.len()
sns.histplot(data=df,x='uzunluk',hue='label',bins=40)
plt.show()


### Boş veri


In [ ]:
df['text']=df['text'].fillna('')
df['title']=df['title'].fillna('')


### Feature Engineering


In [ ]:
df['hepsi']=df['title']+' '+df['text']
df=df.sample(2500,random_state=42)  # tfidf yavaş olmasın
from sklearn.feature_extraction.text import TfidfVectorizer
vec=TfidfVectorizer(max_features=4000,stop_words='english')
x=vec.fit_transform(df['hepsi'].astype(str))
y=df['label']


### Train Test Split


In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)


### 3 Model


In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score,classification_report

for ad,m in [('NB',MultinomialNB()),('LogReg',LogisticRegression(max_iter=200)),('SVC',LinearSVC())]:
    m.fit(x_train,y_train)
    print(ad,accuracy_score(y_test,m.predict(x_test)))


In [ ]:
from sklearn.pipeline import make_pipeline
pipe=make_pipeline(TfidfVectorizer(max_features=4000,stop_words='english'),LinearSVC())
pipe.fit(df['hepsi'].astype(str),y)


In [ ]:
import joblib
joblib.dump(pipe,'../../models/nlp_fake_news.joblib')


### Sonuç

Gerçek fake/real haber setinde LinearSVC iyi ayrım yaptı. Hedefi tutturdum.
